# Σημειώσεις Pandas: Κάθετη Στοίβαξη Πινάκων (`pd.concat()`)

Η συνάρτηση `pd.concat()` χρησιμοποιείται για να **στοιβάξουμε (stack)** δύο ή περισσότερα DataFrames **το ένα κάτω από το άλλο** (κάθετα). Δεν συνδέει πληροφορίες με βάση κάποιο κλειδί (όπως η `merge`), απλώς προσθέτει περισσότερες γραμμές στον πίνακα.

* **`pd.concat()`** ➡️ Προσθέτει **γραμμές** (κάθετη ένωση).
* **`pd.merge()`** ➡️ Προσθέτει **στήλες** (οριζόντια ένωση).

---

## 1. Βασική Σύνταξη & Το `ignore_index=True`

Περνάμε τους πίνακες που θέλουμε να ενώσουμε μέσα σε μια **λίστα `[ ]`**.

```python
import pandas as pd

# Ένωση δύο πινάκων με τις ίδιες στήλες
combined = pd.concat([df_2022, df_2023], ignore_index=True)
```

### ⚠️ Γιατί βάζουμε πάντα `ignore_index=True`;
Κάθε πίνακας ξεκινάει την αρίθμησή του από το `0`. Χωρίς το `ignore_index=True`, ο τελικός πίνακας θα έχει **διπλότυπα index** (π.χ. δύο γραμμές με index `0`, δύο με index `1` κ.ο.κ.). Με το `ignore_index=True`, το Pandas φτιάχνει μια νέα, ενιαία αρίθμηση από το `0` μέχρι το τέλος.

---

## 2. Προσθήκη Ετικέτας Προέλευσης (`source` column)

Όταν ενώνουμε πίνακες (π.χ. πωλήσεις ανά περιοχή ή έτος), χάνεται η πληροφορία από πού προήλθε κάθε γραμμή. Καλή πρακτική είναι να προσθέτουμε μια στήλη-ετικέτα **πριν** την ένωση.

```python
# Χρησιμοποιούμε .copy() για να μην αλλοιώσουμε τους αρχικούς πίνακες
us_copy = us_orders.copy()
us_copy["region"] = "US"

eu_copy = eu_orders.copy()
eu_copy["region"] = "EU"

# Τώρα μπορούμε να τους ενώσουμε και να κάνουμε groupby("region") σε επόμενο βήμα 
all_orders = pd.concat([us_copy, eu_copy], ignore_index=True)
```

---

## 3. Αφαίρεση Διπλότυπων Εγγραφών

Η `pd.concat()` δεν ελέγχει αν υπάρχουν κοινές γραμμές ανάμεσα στους πίνακες· απλώς τους κολλάει. Για να καθαρίσουμε τυχόν διπλότυπα:

```python
combined = pd.concat([df_a, df_b], ignore_index=True).drop_duplicates()
```

---

## 💡 Τι συμβαίνει αν οι στήλες δεν ταιριάζουν;
* Το Pandas **δεν** θα βγάλει error.
* Θα κρατήσει όλες τις στήλες και από τους δύο πίνακες, γεμίζοντας με **`NaN`** (κενές τιμές) τα κελιά όπου η στήλη δεν υπήρχε στον αρχικό πίνακα.


In [1]:
import pandas as pd
import numpy as np

In [2]:
retail = pd.read_csv("../data/retail.csv", index_col=0)

In [3]:
retail.head(3)

,date,category,product,region,channel,units,unit_price,discount_pct,revenue
order_id,,,,,,,,,
ORD-10000,2025-10-15,Books,Puzzle book,East,store,1,178.5,15,178.5
ORD-10001,2025-11-07,Home,Storage box,East,app,1,663.0,15,663.0
ORD-10002,2025-12-26,Stationery,Gel pen pack,East,web,1,90.0,0,90.0


In [4]:
retail = retail.reset_index()

In [5]:
retail['channel'].drop_duplicates()

0    store
1      app
2      web
Name: channel, dtype: str

In [6]:
# Χωρίζω τον πίνακα σε μικρότερους για να κάνω εξάσκηση με το pd.concat()
retail_store = retail[retail['channel']=='store']
retail_app = retail[retail['channel']=='app']
retail_web = retail[retail['channel']=='web']

In [7]:
store_copy = retail_store.copy()
app_copy = retail_app.copy()
web_copy = retail_web.copy()

store_copy['flag'] = 'store'
app_copy['flag'] = 'app'
web_copy['flag'] = 'web'

In [8]:
store_app = pd.concat([store_copy, app_copy], ignore_index=True)

In [9]:
store_app

,order_id,date,category,product,region,channel,units,unit_price,discount_pct,revenue,flag
0,ORD-10000,2025-10-15,Books,Puzzle book,East,store,1,178.5,15,178.5,store
1,ORD-10003,2025-08-02,Home,Desk lamp,North,store,2,1402.5,15,2805.0,store
2,ORD-10006,2025-08-11,Stationery,Notebook A5,East,store,3,120.0,0,360.0,store
3,ORD-10009,2025-11-18,Home,Storage box,South,store,2,663.0,15,1326.0,store
4,ORD-10010,2025-11-02,Home,Ceramic mug,West,store,3,332.5,5,997.5,store
...,...,...,...,...,...,...,...,...,...,...,...
878,ORD-11372,2025-09-28,Home,Ceramic mug,South,app,2,315.0,10,630.0,app
879,ORD-11376,2025-03-26,Electronics,USB-C cable,South,app,5,427.5,5,2137.5,app
880,ORD-11385,2025-09-16,Home,Ceramic mug,West,app,1,315.0,10,315.0,app
881,ORD-11387,2025-02-11,Electronics,USB-C cable,South,app,1,427.5,5,427.5,app


In [10]:
store_app.isna().sum()

order_id        0
date            0
category        0
product         0
region          0
channel         0
units           0
unit_price      0
discount_pct    0
revenue         0
flag            0
dtype: int64

In [11]:
store_app_insights = store_app.groupby('flag').aggregate(sum_revenue=('revenue', 'sum'),
                                                        avg_order=('revenue', 'mean'),
                                                        units_sum=('units','sum'),
                                                        avg_units=('units', 'mean')
                                                        ).round(2)
store_app_insights

,sum_revenue,avg_order,units_sum,avg_units
flag,,,,
app,951482.05,1819.28,1146,2.19
store,494703.40,1374.18,770,2.14


In [12]:
#επιχειρηματική ερμηνεία
#Το app φέρνει σχεδόν τα διπλάσια συνολικά έσοδα
#Οι χρήστες του app αγοράζουν ακριβότερα προϊόντα (avg_order) ανά παραγγελία

In [13]:
pd.crosstab(store_app['flag'], store_app['category'], normalize='index').round(3)*100

category,Books,Electronics,Home,Stationery
flag,,,,
app,14.3,43.4,22.4,19.9
store,19.4,17.5,33.1,30.0


In [14]:
store_app_web = pd.concat([store_copy, web_copy, app_copy], ignore_index=True)

In [16]:
store_app_web['flag'].value_counts()

flag
app      523
web      517
store    360
Name: count, dtype: int64